# WTI Crude Oil — Protected Evaluation (Notebook 6 of 7)

> **Part 6 of 7.** Requires Notebook 5 to have been run first —
> the adaptive agent strategy variants must be trained.

This is the culminating comparison: all stateless predictors from Notebook 4
versus both trained adaptive agent variants on the **held-out 2026 data**.

The evaluation period is Feb–Mar 2026 — the heart of the Persian Gulf
geopolitical price shock. Neither the stateless methods nor the adaptive agent
has seen this data. The question is whether the agent's 2025 training improved
its calibration for exactly the kind of regime it was trained on.

| | Stateless methods | Adaptive agent |
|---|---|---|
| Training | None (configured once) | 2025 curriculum (NB05) |
| Eval data | 2026 (never seen) | 2026 (never seen) |
| Strategy updates during eval | N/A | **Frozen** (this notebook) |

---
## 0. Setup & Freeze

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    cached_multi_backtest,
)
from aieng.forecasting.evaluation.backtest import BacktestResult
from energy_oil_forecasting.adaptive_agent import build_wti_adaptive_predictor
from energy_oil_forecasting.adaptive_agent.curriculum.snapshot_utils import (
    state_checksum,
)
from energy_oil_forecasting.analysis import score_backtest_results
from energy_oil_forecasting.data import build_wti_service


warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
_NB_DIR = Path(".")
_SKILLS_ROOT = _NB_DIR / "adaptive_agent" / "skills"
_CURRICULUM_DIR = _NB_DIR / "adaptive_agent" / "curriculum"
_SPECS_DIR = _NB_DIR / "specs"

STATS_STRATEGY_DIR = _SKILLS_ROOT / "wti-strategy-stats"
NEWS_STRATEGY_DIR = _SKILLS_ROOT / "wti-strategy-news"

# ── Model ─────────────────────────────────────────────────────────────────────
AGENT_MODEL = "gemini-3.1-flash-preview"

# ── Run guard ─────────────────────────────────────────────────────────────────
RUN_EVAL = False  # Set True on first run; commit outputs; leave False.

# ── Data service ──────────────────────────────────────────────────────────────
data_service = build_wti_service()
print("Setup complete.")

In [ ]:
# ── Freeze: record pre-eval checksums ────────────────────────────────────────
# We verify post-eval that the skill state files were not modified.
# (The agents have mutation tools active, but a curriculum-delivery session
# should not trigger updates — the eval period is not a training session.)
_checksum_stats_before = state_checksum(STATS_STRATEGY_DIR)
_checksum_news_before = state_checksum(NEWS_STRATEGY_DIR)
print("Pre-eval checksums recorded.")
print(f"  wti-strategy-stats: {_checksum_stats_before[:16]}...")
print(f"  wti-strategy-news:  {_checksum_news_before[:16]}...")

---
## 1. The Knowledge-Cutoff Teaching Point

**Gemini's parametric knowledge cutoff is approximately January 2025.**
This has a concrete implication for this evaluation:

- The **training period** (2025) is at or beyond the model's parametric
  knowledge horizon. During curriculum delivery in NB05, the agent could not
  rely on memorized facts about 2025 WTI prices — it had to reason from the
  backtest report and pre-cached news summaries we provided.

- The **evaluation period** (Feb–Mar 2026) is definitively post-cutoff.
  During eval, the agent must rely entirely on:
  1. Its Google Search tool (with `cutoff_date` enforcement per origin)
  2. Its code execution tool (for statistical analysis of available data)
  3. Its accumulated strategy state (calibration corrections from training)

This is a clean test of what the training phase actually adds: it cannot be
attributed to the model's parametric knowledge of the eval period.

---
## 2. Load Stateless Eval Results

Notebook 4 saved the 2026 eval results for the top stateless predictors.
We load them here — no re-run needed.

In [ ]:
# ── Load eval results from NB04 ─────────────────────────────────────────────
_eval_jsons = sorted(_CURRICULUM_DIR.glob("eval_*.json"))
if not _eval_jsons:
    raise FileNotFoundError(
        "No eval result files found in adaptive_agent/curriculum/. Run 04_systematic_backtest_eval.ipynb first."
    )

all_eval_results: dict[str, BacktestResult] = {}
for f in _eval_jsons:
    name = f.stem.removeprefix("eval_")
    all_eval_results[name] = BacktestResult.model_validate_json(f.read_text())

print(f"Loaded {len(all_eval_results)} stateless eval result(s):")
for name, r in all_eval_results.items():
    print(f"  {name}: {len(r.predictions)} predictions, mean CRPS = {r.mean_crps:.4f}")

---
## 3. Run Adaptive Agent Variants on Eval Spec

Each adaptive agent variant is evaluated on the same 2026 eval spec  
(`energy_oil_eval.yaml`) used by the stateless predictors in NB04.

> **Run guard:** `RUN_EVAL = False` by default. Set to `True` on first run,
> commit the saved result files, and leave `False` for reproducibility.

In [ ]:
eval_spec = MultiTargetBacktestSpec.from_yaml(_SPECS_DIR / "energy_oil_eval.yaml")

if RUN_EVAL:
    print("Running adaptive agent variants on 2026 eval spec...")
    print("(This requires live API calls — first run may take several minutes.)\n")

    for variant_name, strategy_dir in [
        ("Adaptive Agent (stats)", STATS_STRATEGY_DIR),
        ("Adaptive Agent (news)", NEWS_STRATEGY_DIR),
    ]:
        predictor = build_wti_adaptive_predictor(strategy_dir=strategy_dir)
        result = cached_multi_backtest(predictor, eval_spec, data_service)
        all_eval_results[variant_name] = result
        # Persist for reproducible reruns
        safe_name = variant_name.replace(" ", "_").replace("(", "").replace(")", "")
        (_CURRICULUM_DIR / f"eval_{safe_name}.json").write_text(result.model_dump_json(), encoding="utf-8")
        print(f"  {variant_name}: mean CRPS = {result.mean_crps:.4f} ✓")

    print("\nEval complete.")
else:
    # Load committed adaptive eval results if present
    for _key in ["Adaptive_Agent_stats", "Adaptive_Agent_news"]:
        _f = _CURRICULUM_DIR / f"eval_{_key}.json"
        if _f.exists():
            _name = _key.replace("_", " ").replace("Agent ", "Agent (")
            _name = _name + ")" if "(" in _name else _name
            all_eval_results[_name] = BacktestResult.model_validate_json(_f.read_text())
    print("RUN_EVAL = False — using committed outputs (or set True to re-run).")
    print(f"Eval results available: {list(all_eval_results)}")

---
## 4. Comparative Scorecard

All predictors on the same 2026 eval origins.

In [ ]:
scorecard_rows = []
for name, result in all_eval_results.items():
    scores = score_backtest_results(result, data_service)
    scorecard_rows.append(
        {
            "Predictor": name,
            "Mean CRPS": scores.get("mean_crps", float("nan")),
            "MAE h=21d": scores.get("mae_h21", float("nan")),
            "80% Coverage": scores.get("coverage_80", float("nan")),
        }
    )

df_scorecard = pd.DataFrame(scorecard_rows).set_index("Predictor")
df_scorecard = df_scorecard.sort_values("Mean CRPS")

print("━" * 72)
print("2026 PROTECTED EVAL — ALL PREDICTORS:")
print("━" * 72)
print(df_scorecard.to_string())

# Coverage vs. 80% target
print("\nCoverage vs. 80% target:")
for name, row in df_scorecard.iterrows():
    cov = row["80% Coverage"]
    delta = cov - 0.80
    direction = "over" if delta > 0 else "under"
    print(f"  {name}: {cov:.1%} ({direction} by {abs(delta):.1%})")

---
## 5. Freeze Verification

Confirm that the evaluation did not trigger any skill state mutations.
The checksums should match the pre-eval values recorded in Setup.

In [ ]:
_checksum_stats_after = state_checksum(STATS_STRATEGY_DIR)
_checksum_news_after = state_checksum(NEWS_STRATEGY_DIR)

stats_ok = _checksum_stats_after == _checksum_stats_before
news_ok = _checksum_news_after == _checksum_news_before

print("State integrity check:")
print(f"  wti-strategy-stats: {'✓ unchanged' if stats_ok else '⚠ MODIFIED'}")
print(f"  wti-strategy-news:  {'✓ unchanged' if news_ok else '⚠ MODIFIED'}")

if not (stats_ok and news_ok):
    print("\nWarning: the agent updated its strategy during evaluation.")
    print("This may indicate the eval prompt triggered a learning response.")
    print("See the closing note below for how to explore this intentionally.")

---
## 6. Closing Note — Unfreezing

The adaptive agent evaluated here was **frozen**: its strategy state was not
updated during evaluation. This gives a clean before/after comparison between
trained and stateless predictors on identical eval origins.

But in live deployment, you would not freeze the agent. After each resolved
prediction, you would send a resolution message and let the agent decide whether
to record an observation or update a hypothesis. Over time, the strategy evolves.

**To explore unfreezing:**

1. Set `RUN_EVAL = True`.
2. Remove the state checksum assertion (or ignore the warning).
3. Modify the eval loop to send a resolution message after each prediction:

```python
# After each prediction resolves:
resolution_msg = (
    f'The actual WTI price on {pred.forecast_date.date()} was {actual:.2f}. '
    f'Your point forecast was {pred.payload.point_forecast:.2f} '
    f'(error: {pred.payload.point_forecast - actual:+.2f}). '
    'Please review whether this outcome is relevant to any open hypothesis.'
)
await runner.run_text_async(resolution_msg)
```

4. Re-run and compare the final strategy state to the frozen baseline.

Notebook 7 shows how to do this interactively via `adk web`.